In [1]:
import pandas as pd 
import numpy as np 
import datetime as dt 
import json
from pprint import pprint

In [2]:
df = pd.read_csv("AMLNet_August 2025.csv")

In [3]:
df.head()

,step,type,amount,category,nameOrig,nameDest,oldbalanceOrg,newbalanceOrig,isFraud,isMoneyLaundering,laundering_typology,metadata,fraud_probability,hour,day_of_week,day_of_month,month
0,0,DEBIT,298.842041,Other,C8083,C7053,455489.321571,455190.479531,0,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN,12,1,4,2
1,0,DEBIT,93.087916,Recreation,C5575,C1117,229508.291214,229415.203298,0,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN,12,1,4,2
2,0,EFTPOS,155.644864,Healthcare,C1549,C1423,202568.806856,202413.161992,0,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN,12,1,4,2
3,0,BPAY,299.759073,Food,C7435,C6390,491560.600203,491260.841131,0,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN,12,1,4,2
4,0,DEBIT,173.715615,Other,C8083,C5946,455190.479531,455016.763916,0,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN,12,1,4,2


In [4]:
# isFraud and isMoneyLaundering are equivalent
(df["isFraud"] != df["isMoneyLaundering"]).sum()

np.int64(0)

In [5]:
# oldBalance - newBalance = amount 
(((df["oldbalanceOrg"] - df["newbalanceOrig"]) - df["amount"]) > 0.01).sum()

np.int64(0)

In [6]:
df = df.drop(columns = ["hour", "day_of_week", "day_of_month", "month", "amount", "isMoneyLaundering", "step"])
df = df.rename(columns = {"nameOrig" : "orig", "nameDest" : "dest", "oldbalanceOrg" : "oldBalance", "newbalanceOrig" : "newBalance"})
df.head()

,type,category,orig,dest,oldBalance,newBalance,isFraud,laundering_typology,metadata,fraud_probability
0,DEBIT,Other,C8083,C7053,455489.321571,455190.479531,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN
1,DEBIT,Recreation,C5575,C1117,229508.291214,229415.203298,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN
2,EFTPOS,Healthcare,C1549,C1423,202568.806856,202413.161992,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN
3,BPAY,Food,C7435,C6390,491560.600203,491260.841131,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN
4,DEBIT,Other,C8083,C5946,455190.479531,455016.763916,0,normal,"{'timestamp': datetime.datetime(2025, 2, 4, 12...",NaN


In [7]:
pprint(df["metadata"].iloc[0])

("{'timestamp': datetime.datetime(2025, 2, 4, 12, 22, 36, 770518), 'location': "
 "{'city': 'Sydney', 'state': 'NSW', 'country': 'Australia', 'postcode': "
 "'2755'}, 'device_info': {'type': 'Mobile', 'os': 'Windows', 'ip_address': "
 "'109.75.48.170'}, 'payment_method': 'BSB_Account', 'merchant_info': None, "
 "'risk_indicators': {'amount_vs_average': 2.403414656867827, "
 "'customer_risk_score': 100, 'category_risk': 'medium', 'risk_score': "
 "66.30275436133387, 'unusual_time': False, 'unusual_location': False}}")


In [8]:
clean_str = (
    df["metadata"]
    # Convert datetime.datetime(...) into valid JSON string "YYYY-MM-DD HH:MM:SS"
    .str.replace(
        r"datetime\.datetime\((\d+),\s*(\d+),\s*(\d+),\s*(\d+),\s*(\d+),\s*(\d+)[^)]*\)",
        r'"\1-\2-\3 \4:\5:\6"',
        regex=True,
    )
    # Single quotes -> double quotes
    .str.replace("'", '"')
    # Python literals -> JSON standard literals
    .str.replace("None", "null")
    .str.replace("True", "true")
    .str.replace("False", "false")
)

In [9]:
meta_df = pd.json_normalize(clean_str.apply(json.loads))

In [10]:
meta_df.head()

,timestamp,payment_method,merchant_info,location.city,location.state,location.country,location.postcode,device_info.type,device_info.os,device_info.ip_address,...,integration_info.total_amount,integration_info.num_sources,integration_info.average_amount,structuring.sophistication,structuring.threshold_proximity,structuring.pattern_size,layering,layering_sophistication,integration_info.sector,integration_info.platform
0,2025-2-4 12:22:36,BSB_Account,NaN,Sydney,NSW,Australia,2755,Mobile,Windows,109.75.48.170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-2-4 12:44:31,CardNumber,NaN,Brisbane,QLD,Australia,4976,Web,Android,208.137.250.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-2-4 12:48:1,PayID,NaN,Sydney,NSW,Australia,2927,Mobile,Windows,183.206.224.165,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-2-4 12:57:48,CardNumber,NaN,Sydney,NSW,Australia,2332,Mobile,MacOS,252.196.170.132,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-2-4 12:58:37,CardNumber,NaN,Sydney,NSW,Australia,2572,Web,Windows,113.123.44.252,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
meta_df["datetime"] = pd.to_datetime(meta_df["timestamp"])
meta_df = meta_df.drop(columns = ["timestamp"])

In [12]:
meta_df.columns

Index(['payment_method', 'merchant_info', 'location.city', 'location.state',
       'location.country', 'location.postcode', 'device_info.type',
       'device_info.os', 'device_info.ip_address',
       'risk_indicators.amount_vs_average',
       'risk_indicators.customer_risk_score', 'risk_indicators.category_risk',
       'risk_indicators.risk_score', 'risk_indicators.unusual_time',
       'risk_indicators.unusual_location', 'merchant_info.merchant_id',
       'merchant_info.category', 'merchant_info.risk_level',
       'merchant_info.avg_transaction', 'sophistication',
       'integration_info.type', 'integration_info.legitimacy_score',
       'integration_info.detection_risk', 'integration_info.location',
       'integration_info.total_amount', 'integration_info.num_sources',
       'integration_info.average_amount', 'structuring.sophistication',
       'structuring.threshold_proximity', 'structuring.pattern_size',
       'layering', 'layering_sophistication', 'integration_info.sec

In [13]:
# Drop the columns with over 90% of the values missing
cols_to_drop = [col for col in meta_df.columns if meta_df[col].isna().mean() > 0.9]
meta_df = meta_df.drop(columns = cols_to_drop)
print(cols_to_drop)
meta_df.columns

['merchant_info', 'sophistication', 'integration_info.type', 'integration_info.legitimacy_score', 'integration_info.detection_risk', 'integration_info.location', 'integration_info.total_amount', 'integration_info.num_sources', 'integration_info.average_amount', 'structuring.sophistication', 'structuring.threshold_proximity', 'structuring.pattern_size', 'layering', 'layering_sophistication', 'integration_info.sector', 'integration_info.platform']


Index(['payment_method', 'location.city', 'location.state', 'location.country',
       'location.postcode', 'device_info.type', 'device_info.os',
       'device_info.ip_address', 'risk_indicators.amount_vs_average',
       'risk_indicators.customer_risk_score', 'risk_indicators.category_risk',
       'risk_indicators.risk_score', 'risk_indicators.unusual_time',
       'risk_indicators.unusual_location', 'merchant_info.merchant_id',
       'merchant_info.category', 'merchant_info.risk_level',
       'merchant_info.avg_transaction', 'datetime'],
      dtype='object')

In [14]:
# Location preprocessing
loc_cols = [c for c in meta_df.columns if c.startswith("location.")]
meta_df[loc_cols].nunique()

location.city           6
location.state          5
location.country        1
location.postcode    5000
dtype: int64

In [15]:
meta_df = meta_df.drop(columns = ["location.country"])
meta_df.head()

,payment_method,location.city,location.state,location.postcode,device_info.type,device_info.os,device_info.ip_address,risk_indicators.amount_vs_average,risk_indicators.customer_risk_score,risk_indicators.category_risk,risk_indicators.risk_score,risk_indicators.unusual_time,risk_indicators.unusual_location,merchant_info.merchant_id,merchant_info.category,merchant_info.risk_level,merchant_info.avg_transaction,datetime
0,BSB_Account,Sydney,NSW,2755,Mobile,Windows,109.75.48.170,2.403415,100.0,medium,66.302754,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:22:36
1,CardNumber,Brisbane,QLD,4976,Web,Android,208.137.250.85,2.094116,100.0,medium,62.167417,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:44:31
2,PayID,Sydney,NSW,2927,Mobile,Windows,183.206.224.165,0.525958,100.0,low,49.266092,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:48:01
3,CardNumber,Sydney,NSW,2332,Mobile,MacOS,252.196.170.132,1.094962,100.0,low,49.602450,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:57:48
4,CardNumber,Sydney,NSW,2572,Web,Windows,113.123.44.252,1.397095,100.0,medium,59.465938,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:58:37


In [16]:
df = df.drop(columns=["metadata"]).join(meta_df)
df.head()

,type,category,orig,dest,oldBalance,newBalance,isFraud,laundering_typology,fraud_probability,payment_method,...,risk_indicators.customer_risk_score,risk_indicators.category_risk,risk_indicators.risk_score,risk_indicators.unusual_time,risk_indicators.unusual_location,merchant_info.merchant_id,merchant_info.category,merchant_info.risk_level,merchant_info.avg_transaction,datetime
0,DEBIT,Other,C8083,C7053,455489.321571,455190.479531,0,normal,NaN,BSB_Account,...,100.0,medium,66.302754,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:22:36
1,DEBIT,Recreation,C5575,C1117,229508.291214,229415.203298,0,normal,NaN,CardNumber,...,100.0,medium,62.167417,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:44:31
2,EFTPOS,Healthcare,C1549,C1423,202568.806856,202413.161992,0,normal,NaN,PayID,...,100.0,low,49.266092,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:48:01
3,BPAY,Food,C7435,C6390,491560.600203,491260.841131,0,normal,NaN,CardNumber,...,100.0,low,49.602450,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:57:48
4,DEBIT,Other,C8083,C5946,455190.479531,455016.763916,0,normal,NaN,CardNumber,...,100.0,medium,59.465938,False,False,NaN,NaN,NaN,NaN,2025-02-04 12:58:37


In [17]:
df.to_csv("dataset.csv")